# Evolvable Policies — Validation Notebook

Replicates the core result from **Thoma, Vassiliades & Michael (2026)**:

> Starting from an empty policy and random neural weights, a NeSy organism discovers
> a symbolic policy that approximates a hidden target policy through evolutionary search.

The system uses **no gradient signal** for the symbolic component — rules are learned
via mutation (S+/S−) and fitness selection, not backpropagation.

**Parameters are kept small so this finishes in ~15–25 minutes on CPU (no GPU needed).**

## 1. Setup

In [ ]:
import os

REPO_URL = 'https://github.com/konstantinoslamp/NeuroSymbolic-Network-for-Handwritten-Symbol-Understanding.git'
REPO_DIR = '/content/neurosymbolic_mvp'

if os.path.exists(REPO_DIR):
    print('Repo exists, pulling latest...')
    os.chdir(REPO_DIR)
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q numpy Pillow tqdm
import sys
sys.path.insert(0, REPO_DIR)

# Sanity-check import
from src.evolvable.run_experiment import run_single_experiment, save_results
print('Import OK.')

## 2. Experiment Parameters

Small values so the run completes in ~15–25 min.  
Increase `NUM_ATOMS`, `GENERATIONS`, `TRAIN_EPOCHS` for stronger results.

In [ ]:
CONFIG = {
    'num_atoms':           3,     # number of propositional atoms (a1, a2, a3)
    'num_rules':           3,     # rules in the hidden target policy
    'policy_seed':        42,     # reproducibility
    'data_seed':        1042,
    'train_size':        200,     # training samples per generation
    'val_size':           80,
    'test_size':          80,
    'train_epochs':        5,     # epochs to train each organism
    'learning_rate':    0.001,
    'max_generations':    15,     # evolutionary generations
    'early_stop_accuracy': 0.97,  # stop early if reached
    'mnist_path':        None,    # auto-downloads if None
    'verbose':           True,
}

print('Config:')
for k, v in CONFIG.items():
    print(f'  {k:25s} = {v}')

## 3. Run Evolutionary Experiment

The pipeline:
1. Generate a random **hidden target policy** (3 rules over atoms a1, a2, a3)
2. Build an MNIST-based dataset labelled by the target policy
3. Start with an organism with **empty policy + random CNN weights**
4. Each generation: produce 6 offspring (3 symbolic mutations × 2 neural mutations),
   train each, measure fitness vs parent, select next parent
5. Track generational accuracy — the key claim is that it rises toward ~97–99%

In [ ]:
import time
t0 = time.time()

results = run_single_experiment(CONFIG)

print(f'\nTotal time: {time.time() - t0:.1f}s')

## 4. Results Summary

In [ ]:
summary = results['summary']
ev      = results['evolution_result']

print('=' * 55)
print('  EVOLVABLE POLICIES — RESULTS')
print('=' * 55)
print(f'  Generations run    : {summary["num_generations"]}')
print(f'  Initial accuracy   : {summary["initial_accuracy"]:.3f}')
print(f'  Final accuracy     : {summary["final_accuracy"]:.3f}')
print(f'  Best val accuracy  : {summary["best_accuracy"]:.3f}')
print(f'  Final wrong rate   : {summary["final_wrong_rate"]:.3f}')
print(f'  Final abstain rate : {summary["final_abstain_rate"]:.3f}')
print()
print(f'  Selection counts   : {summary["selection_counts"]}')

if ev.get('final_test'):
    t = ev['final_test']
    print()
    print('  TEST SET:')
    print(f'    Correct  : {t["correct"]:.3f}')
    print(f'    Wrong    : {t["wrong"]:.3f}')
    print(f'    Abstain  : {t["abstain"]:.3f}')
print('=' * 55)

print()
print('Hidden target policy:')
print(results['target_policy'])

## 5. Generational Fitness Trajectory

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

lineage = ev.get('lineage', [])

if lineage:
    gens     = [g for g, _, _ in lineage]
    fitnesses = [f for _, _, f in lineage]
    labels   = [l for _, l, _ in lineage]

    # Map mutation labels to colours
    colour_map = {
        'S0+Npw': '#95a5a6', 'S0+Nrw': '#bdc3c7',
        'S++Npw': '#2ecc71', 'S++Nrw': '#27ae60',
        'S-+Npw': '#e74c3c', 'S-+Nrw': '#c0392b',
    }
    colours = [colour_map.get(l, '#3498db') for l in labels]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: fitness per generation (bar chart with mutation colour)
    ax = axes[0]
    bars = ax.bar(gens, fitnesses, color=colours, edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Generation')
    ax.set_ylabel('Fitness (vs parent)')
    ax.set_title('Relative Fitness per Generation')
    ax.grid(True, alpha=0.3, axis='y')

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, label=l) for l, c in colour_map.items()]
    ax.legend(handles=legend_elements, fontsize=7, loc='upper left')

    # Right: cumulative best val accuracy over generations
    ax = axes[1]
    # Reconstruct val accuracy from lineage (fitness is relative, not absolute)
    # Use cumulative max of fitness as proxy for improvement
    cum_fitness = np.maximum.accumulate(fitnesses)
    ax.plot(gens, fitnesses,     color='#3498db', linewidth=1.5,
            marker='o', markersize=4, label='Fitness', alpha=0.7)
    ax.plot(gens, cum_fitness,   color='#e74c3c', linewidth=2,
            linestyle='--', label='Cumulative max')
    ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
    ax.set_xlabel('Generation')
    ax.set_ylabel('Fitness')
    ax.set_title('Fitness Trajectory')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('evolvable_fitness.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: evolvable_fitness.png')
else:
    print('No lineage data to plot.')

## 6. Accuracy Over Generations

In [ ]:
# Rebuild accuracy trajectory from the history stored in evolution_result
# The EvolutionaryEngine stores per-generation accuracy in the history list
history = results.get('evolution_result', {}).get('history', [])

# Fallback: construct from summary if full history not present
if not history:
    print('Full history not in results — showing initial/final only.')
    s = results['summary']
    print(f"  Initial accuracy: {s['initial_accuracy']:.3f}")
    print(f"  Final accuracy:   {s['final_accuracy']:.3f}")
    print(f"  Best accuracy:    {s['best_accuracy']:.3f}")
else:
    gen_nums  = [h['generation']  for h in history]
    val_accs  = [h.get('val_correct', h.get('accuracy', 0)) for h in history]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(gen_nums, val_accs, color='#2ecc71', linewidth=2,
            marker='o', markersize=5, label='Val accuracy')
    ax.axhline(results['summary']['best_accuracy'], color='#e74c3c',
               linestyle='--', linewidth=1.5, label=f'Best ({results["summary"]["best_accuracy"]:.3f})')
    ax.set_xlabel('Generation')
    ax.set_ylabel('Validation Accuracy')
    ax.set_title('Accuracy Across Evolutionary Generations')
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('evolvable_accuracy.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: evolvable_accuracy.png')

## 7. Evolved Policy vs Target Policy

In [ ]:
print('Hidden TARGET policy (what the system had to discover):')
print('─' * 50)
print(results['target_policy'])
print()

lineage = results['evolution_result'].get('lineage', [])
if lineage:
    print(f'Lineage — mutation chosen each generation:')
    print('─' * 50)
    print(f'{"Gen":>4}  {"Mutation":<12}  {"Fitness":>8}')
    print('─' * 30)
    for gen, label, fitness in lineage:
        marker = ' ✓' if fitness > 0 else ('  ' if fitness == 0 else ' ✗')
        print(f'{gen:>4}  {label:<12}  {fitness:>+8.3f}{marker}')
    print()

    # Count mutation type outcomes
    from collections import Counter
    counts = Counter(l for _, l, _ in lineage)
    print('Mutation type selection frequency:')
    for label, count in sorted(counts.items(), key=lambda x: -x[1]):
        print(f'  {label:<12}: {count}')

## 8. Paper-Ready Summary

In [ ]:
s  = results['summary']
ev = results['evolution_result']

print('\n' + '='*60)
print('  EVOLVABLE POLICIES — PAPER TABLE')
print('='*60)
print(f'  Config: {CONFIG["num_atoms"]} atoms, {CONFIG["num_rules"]} target rules, '
      f'{CONFIG["max_generations"]} generations')
print()
print(f'  Initial accuracy : {s["initial_accuracy"]:.3f}  (random init)')
print(f'  Final accuracy   : {s["final_accuracy"]:.3f}')
print(f'  Best accuracy    : {s["best_accuracy"]:.3f}')
print(f'  Generations      : {s["num_generations"]}')
if ev.get('final_test'):
    t = ev['final_test']
    print(f'  Test correct     : {t["correct"]:.3f}')
    print(f'  Test wrong       : {t["wrong"]:.3f}')
    print(f'  Test abstain     : {t["abstain"]:.3f}')
print('='*60)

print()
print('LaTeX snippet:')
print(r'\begin{table}[h]\centering')
print(r'\begin{tabular}{lc}')
print(r'\toprule')
print(r'Metric & Value \\')
print(r'\midrule')
print(f'Initial accuracy & {s["initial_accuracy"]:.3f} \\\\')
print(f'Final accuracy   & {s["final_accuracy"]:.3f} \\\\')
print(f'Best accuracy    & {s["best_accuracy"]:.3f} \\\\')
print(f'Generations      & {s["num_generations"]} \\\\')
if ev.get('final_test'):
    print(f'Test accuracy    & {ev["final_test"]["correct"]:.3f} \\\\')
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\caption{Evolvable Policies: accuracy over evolutionary generations.}')
print(r'\label{tab:evolvable}')
print(r'\end{table}')

## 9. Save & Download Results

In [ ]:
os.makedirs('results', exist_ok=True)
output_path = 'results/evolvable_results.json'
save_results(results, output_path)
print(f'Saved {os.path.getsize(output_path)/1024:.1f} KB → {output_path}')

try:
    from google.colab import files
    files.download(output_path)
    if os.path.exists('evolvable_fitness.png'):
        files.download('evolvable_fitness.png')
    if os.path.exists('evolvable_accuracy.png'):
        files.download('evolvable_accuracy.png')
    print('Downloads triggered.')
except ImportError:
    print('Not in Colab — files saved locally.')